# SROIE OCR Error Analysis

This notebook evaluates whether validated SROIE ground-truth values are recoverable from the existing Tesseract OCR output. It uses substring matching after lightweight evaluation-only normalization; it does not extract fields, alter OCR text, modify raw data, or train a model.

In [1]:
import json
import re
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.sroie import load_sroie_dataset

DATASET_ROOT = PROJECT_ROOT / 'data' / 'raw'
OCR_PATH = PROJECT_ROOT / 'data' / 'processed' / 'sroie_ocr.jsonl'
FIELDS = ['company', 'date', 'address', 'total']

dataset = load_sroie_dataset(DATASET_ROOT, strict=False)
print(f'Validated SROIE samples: {len(dataset)}')
print(f'OCR results path: {OCR_PATH}')
print(f'OCR results available: {OCR_PATH.is_file()}')

Validated SROIE samples: 624
OCR results path: c:\Users\chech\Documents\AI-Document-Intelligence\data\processed\sroie_ocr.jsonl
OCR results available: False


## Load OCR results

The loader below reads each JSONL record without rewriting its `text` value. Joining is by the stable `image_id` emitted by the existing batch OCR utility.

In [2]:
def load_ocr_jsonl(path: Path) -> pd.DataFrame:
    records = []
    with path.open(encoding='utf-8') as stream:
        for line_number, line in enumerate(stream, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            if not isinstance(record, dict):
                raise ValueError(f'OCR line {line_number} is not a JSON object')
            records.append(record)
    return pd.DataFrame.from_records(records)

if OCR_PATH.is_file():
    ocr_results = load_ocr_jsonl(OCR_PATH)
    required_ocr_columns = {'image_id', 'text'}
    missing_ocr_columns = required_ocr_columns - set(ocr_results.columns)
    if missing_ocr_columns:
        raise ValueError(f'OCR JSONL is missing columns: {sorted(missing_ocr_columns)}')
    print(f'OCR records loaded: {len(ocr_results)}')
else:
    ocr_results = pd.DataFrame(columns=['image_id', 'text', 'status'])
    print('No full OCR JSONL found. Run the baseline batch OCR first; analysis tables will be empty.')

ocr_by_id = ocr_results.drop_duplicates('image_id', keep='first').set_index('image_id')
analysis = dataset[['image_id', *FIELDS]].copy()
analysis['ocr_text'] = analysis['image_id'].map(ocr_by_id['text']).fillna('')
if 'status' in ocr_by_id:
    analysis['ocr_status'] = analysis['image_id'].map(ocr_by_id['status']).fillna('missing')
else:
    analysis['ocr_status'] = 'unknown'

assert len(analysis) == len(dataset)
print(f'Joined analysis rows: {len(analysis)}')
display(analysis[['image_id', 'ocr_status']].head())

No full OCR JSONL found. Run the baseline batch OCR first; analysis tables will be empty.
Joined analysis rows: 624


,image_id,ocr_status
0,X00016469612,missing
1,X00016469619,missing
2,X00016469620,missing
3,X00016469622,missing
4,X00016469623,missing


## Evaluation-only normalization

Normalization removes formatting noise only: case, whitespace/line breaks, and the decimal separator in numeric text. The original ground truth and OCR text remain in `analysis` unchanged. No regex-based field extraction is used.

In [3]:
def normalize_for_match(value: object) -> str:
    text = str(value) if value is not None else ''
    text = text.casefold().replace(',', '.')
    return re.sub(r'\s+', '', text)

def contains_ground_truth(ground_truth: object, ocr_text: object) -> bool:
    normalized_truth = normalize_for_match(ground_truth)
    normalized_ocr = normalize_for_match(ocr_text)
    return bool(normalized_truth) and normalized_truth in normalized_ocr

for field in FIELDS:
    analysis[f'{field}_valid'] = analysis[field].fillna('').astype(str).str.strip().ne('')
    analysis[f'{field}_match'] = [
        contains_ground_truth(truth, ocr) if valid else False
        for truth, ocr, valid in zip(analysis[field], analysis['ocr_text'], analysis[f'{field}_valid'])
    ]

assert analysis['ocr_text'].equals(analysis['image_id'].map(ocr_by_id['text']).fillna(''))
print('Original OCR text preserved: yes')

Original OCR text preserved: yes


## Coverage results

In [4]:
coverage_rows = []
for field in FIELDS:
    valid_count = int(analysis[f'{field}_valid'].sum())
    matched_count = int(analysis[f'{field}_match'].sum())
    coverage_rows.append({
        'field': field,
        'valid_samples': valid_count,
        'normalized_ground_truth_found': matched_count,
        'coverage_percent': round(100 * matched_count / valid_count, 2) if valid_count else 0.0,
    })

coverage = pd.DataFrame(coverage_rows).set_index('field')
display(coverage)

,valid_samples,normalized_ground_truth_found,coverage_percent
field,,,
company,624,0,0.0
date,624,0,0.0
address,624,0,0.0
total,624,0,0.0


## Representative matches and mismatches

Examples show the original OCR text exactly as loaded, including line breaks and OCR errors.

In [5]:
def show_examples(field: str, matched: bool, limit: int = 3) -> None:
    subset = analysis[(analysis[f'{field}_valid']) & (analysis[f'{field}_match'] == matched)]
    columns = ['image_id', field, 'ocr_status', 'ocr_text']
    title = 'successful matches' if matched else 'OCR mismatches'
    print(f'\n{field}: {title} ({len(subset)} available)')
    display(subset[columns].head(limit).style.set_properties(subset=['ocr_text'], **{'white-space': 'pre-wrap'}))

for field in FIELDS:
    show_examples(field, matched=True)
    show_examples(field, matched=False)


company: successful matches (0 available)


,image_id,company,ocr_status,ocr_text



company: OCR mismatches (624 available)


,image_id,company,ocr_status,ocr_text
0,X00016469612,BOOK TA .K (TAMAN DAYA) SDN BHD,missing,
1,X00016469619,INDAH GIFT & HOME DECO,missing,
2,X00016469620,MR D.I.Y. (JOHOR) SDN BHD,missing,



date: successful matches (0 available)


...

,image_id,address,ocr_status,ocr_text
0,X00016469612,"NO.53 55,57 & 59, JALAN SAGU 18, TAMAN DAYA, 81100 JOHOR BAHRU, JOHOR.",missing,
1,X00016469619,"27, JALAN DEDAP 13, TAMAN JOHOR JAYA, 81100 JOHOR BAHRU, JOHOR.",missing,
2,X00016469620,"LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERINDUSTRIAN BALAKONG, 43300 SERI KEMBANGAN, SELANGOR (MR DIY TESCO TERBAU)",missing,



total: successful matches (0 available)


,image_id,total,ocr_status,ocr_text



total: OCR mismatches (624 available)


,image_id,total,ocr_status,ocr_text
0,X00016469612,9.00,missing,
1,X00016469619,60.30,missing,
2,X00016469620,33.90,missing,


In [6]:
if OCR_PATH.is_file() and not coverage.empty:

    lowest_coverage = coverage['coverage_percent'].sort_values().index.tolist()

    print('Fields most affected by normalized substring misses: ' + ', '.join(lowest_coverage))

    print('Next preprocessing experiments: grayscale/contrast normalization, light denoising, adaptive thresholding, deskewing, border removal or receipt cropping, and selected Tesseract page-segmentation modes.')

else:

    print('No field-level OCR conclusion is reported because data/processed/sroie_ocr.jsonl is unavailable.')

    print('Next preprocessing experiments: grayscale/contrast normalization, light denoising, adaptive thresholding, deskewing, border removal or receipt cropping, and selected Tesseract page-segmentation modes.')

No field-level OCR conclusion is reported because data/processed/sroie_ocr.jsonl is unavailable.
Next preprocessing experiments: grayscale/contrast normalization, light denoising, adaptive thresholding, deskewing, border removal or receipt cropping, and selected Tesseract page-segmentation modes.


## Interpretation and next experiments

Use the coverage table and examples above to identify the lowest-coverage fields. Typical failure patterns to inspect are character substitutions in company names, punctuation or line-layout damage in addresses, date separator/character confusion, and decimal/comma confusion in totals.

The next preprocessing experiments should be isolated and measured against this baseline: grayscale conversion and contrast normalization, light denoising, adaptive thresholding, deskewing, receipt cropping or border removal, and a small set of Tesseract page-segmentation modes. Re-run the same comparison after each experiment while preserving the original OCR JSONL for auditability.